In [87]:
import pandas as pd
df=pd.read_csv('/Users/harsha/RAG/N/dtst.csv',sep=';')
df.head()

,mac_rnti,mac_dl_cqi,mac_dl_mcs,mac_dl_brate,mac_dl_ok,mac_dl_nok,phy_ul_pusch_sinr,phy_ul_pucch_sinr,phy_ul_mcs,mac_ul_brate,...,phy_dl_n_samples,rf_o,rf_u,rf_l,rf_error,label,ue_ident,timestamp,id_ue,mob_pattern
0,3540,8,14,34577,1,0,27.973768,29.404709,20.250000,360977,...,7,0,0,0,0,portscan,3540,1.666859e+14,1,car
1,3540,6,12,1034844,16,6,25.914433,30.450718,18.360001,3849777,...,70,0,0,0,0,portscan,3540,1.666859e+14,1,car
2,3540,7,11,5538,1,0,30.937387,30.019136,13.142858,49494,...,3,0,0,0,0,portscan,3540,1.666859e+14,1,car
3,3540,8,12,33155,1,0,31.344568,29.917725,19.600000,153244,...,5,0,0,0,0,portscan,3540,1.666859e+14,1,car
4,3540,5,11,1196444,14,12,22.785067,30.253895,19.829546,4017600,...,62,0,0,0,0,portscan,3540,1.666859e+14,1,car


In [ ]:
print(df.shape)
print(df.head())
print(df.dtypes)

(3175140, 47)
   mac_rnti  mac_dl_cqi  mac_dl_mcs  mac_dl_brate  mac_dl_ok  mac_dl_nok  \
0      3540           8          14         34577          1           0   
1      3540           6          12       1034844         16           6   
2      3540           7          11          5538          1           0   
3      3540           8          12         33155          1           0   
4      3540           5          11       1196444         14          12   

   phy_ul_pusch_sinr  phy_ul_pucch_sinr  phy_ul_mcs  mac_ul_brate  ...  \
0          27.973768          29.404709   20.250000        360977  ...   
1          25.914433          30.450718   18.360001       3849777  ...   
2          30.937387          30.019136   13.142858         49494  ...   
3          31.344568          29.917725   19.600000        153244  ...   
4          22.785067          30.253895   19.829546       4017600  ...   

   phy_dl_n_samples  rf_o  rf_u  rf_l  rf_error     label  ue_ident  \
0            

In [89]:
df = df.dropna()

## these columns are deleted because this are explicit id's,not an usefull data

In [ ]:
drop_cols = ["ue_ident","id_ue","timestamp"]
df = df.drop(columns=drop_cols)

In [ ]:
print(df["label"].value_counts())

label
dos-hulk-C       564008
iot              508187
ddos-ripper-C    490425
portscan         461151
SIPP             383769
youtube          319119
Web Browsing     238140
slowloris-C      210341
Name: count, dtype: int64


## As mob_pattern contains 3 categories ,so we are moving names to numbers 

In [ ]:
import sklearn
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["mob_pattern"] = le.fit_transform( df["mob_pattern"])

## As these columns contain only onr type of data ,so drop

In [ ]:
constant_cols = [c for c in df.columns if df[c].nunique() <= 1]
print(constant_cols)

['mac_pci', 'mac_cc_idx', 'mac_dl_ri', 'mac_dl_pmi', 'mac_ul_rssi', 'mac_fec_iters', 'mac_dl_mcs_samples', 'mac_ul_mcs', 'mac_ul_mcs_samples', 'phy_ul_n', 'phy_ul_pusch_tpc', 'phy_dl_pucch_tpc', 'rf_o', 'rf_u', 'rf_l']


In [ ]:
df = df.drop( columns=constant_cols)

## check correlation between atributes,if correlation between attributes >=95% then drop those columns,as taking onre ogf them wil be representing both columns.

In [ ]:
corr = (df.select_dtypes(include="number").corr().abs())

## As take only upper trainagular matrix in correlation matrix

In [ ]:
import numpy as np
upper = corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool))

## then drop the columsn 

In [ ]:
to_drop = [column for column in upper.columns if any( upper[column] > 0.95)]
df = df.drop(columns=to_drop)
print(to_drop)

['phy_ul_pucch_ni', 'phy_dl_mcs']


In [98]:
df.head()

,mac_rnti,mac_dl_cqi,mac_dl_mcs,mac_dl_brate,mac_dl_ok,mac_dl_nok,phy_ul_pusch_sinr,phy_ul_pucch_sinr,phy_ul_mcs,mac_ul_brate,...,mac_ul_snr_offset,phy_ul_pusch_rssi,phy_ul_pucch_rssi,phy_ul_turbo_iters,phy_ul_n_samples,phy_ul_n_samples_pucch,phy_dl_n_samples,rf_error,label,mob_pattern
0,3540,8,14,34577,1,0,27.973768,29.404709,20.250000,360977,...,0.008000,-47.570415,-42.019466,2.000000,8,7,7,0,portscan,1
1,3540,6,12,1034844,16,6,25.914433,30.450718,18.360001,3849777,...,-0.037000,-47.220612,-42.024117,2.480000,75,21,70,0,portscan,1
2,3540,7,11,5538,1,0,30.937387,30.019136,13.142858,49494,...,-2.200003,-38.277866,-42.004459,2.000000,7,13,3,0,portscan,1
3,3540,8,12,33155,1,0,31.344568,29.917725,19.600000,153244,...,-2.195004,-45.934284,-42.007332,2.000000,5,10,5,0,portscan,1
4,3540,5,11,1196444,14,12,22.785067,30.253895,19.829546,4017600,...,-2.335005,-52.582420,-42.021011,3.238636,88,44,62,0,portscan,1


In [ ]:
print(df["label"].value_counts())

label
dos-hulk-C       564008
iot              508187
ddos-ripper-C    490425
portscan         461151
SIPP             383769
youtube          319119
Web Browsing     238140
slowloris-C      210341
Name: count, dtype: int64


In [ ]:
attack_labels = {"dos-hulk-C","ddos-ripper-C","slowloris-C","portscan"}

df["anomaly"] = ( df["label"].isin(attack_labels).astype(int))

In [101]:
df["anomaly"].value_counts()

anomaly
1    1725925
0    1449215
Name: count, dtype: int64

In [102]:
X=df.drop(columns=['label','anomaly'])
Y=df['anomaly']

In [103]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,random_state=42,test_size=0.2,stratify=Y)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [ ]:
from xgboost import XGBClassifier
model = XGBClassifier(
    n_estimators=300,
    max_depth=15,
    learning_rate=0.1,
    min_child_weight=1,
    gamma=0,
    subsample=1.0,
    colsample_bytree=1.0,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42
)
model.fit(X_scaled, Y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,1.0
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'auc'


## All the metrics are giving 1.0,model is performing very well

In [ ]:
from sklearn.metrics import classification_report
pred = model.predict(X_test_scaled)
print(classification_report( Y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    289843
           1       1.00      1.00      1.00    345185

    accuracy                           1.00    635028
   macro avg       1.00      1.00      1.00    635028
weighted avg       1.00      1.00      1.00    635028



In [ ]:
train_acc = model.score(X_scaled, Y_train)
test_acc = model.score(X_test_scaled, Y_test)
print(train_acc, test_acc)

0.9999649621748963 0.9965907015123743


In [ ]:
import joblib
joblib.dump(model,"model.pkl")

joblib.dump(scaler,"scaler.pkl")

joblib.dump(le,"label_encoder.pkl")

['label_encoder.pkl']

In [109]:
print(X.columns)

Index(['mac_rnti', 'mac_dl_cqi', 'mac_dl_mcs', 'mac_dl_brate', 'mac_dl_ok',
       'mac_dl_nok', 'phy_ul_pusch_sinr', 'phy_ul_pucch_sinr', 'phy_ul_mcs',
       'mac_ul_brate', 'mac_ul_ok', 'mac_ul_nok', 'mac_ul_bsr', 'mac_nof_tti',
       'mac_dl_buffer', 'mac_phr', 'mac_dl_cqi_offset', 'mac_ul_snr_offset',
       'phy_ul_pusch_rssi', 'phy_ul_pucch_rssi', 'phy_ul_turbo_iters',
       'phy_ul_n_samples', 'phy_ul_n_samples_pucch', 'phy_dl_n_samples',
       'rf_error', 'mob_pattern'],
      dtype='object')


## we are checking importance of each feature after tarining ,so that what features matters more /on which faetures value the model is more dependent.

In [ ]:
import pandas as pd
importances = model.feature_importances_
X = pd.DataFrame(X)
feature_names=X.columns
importance_mapping = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)
print(importance_mapping)


                   Feature  Importance
0             mac_ul_brate    0.681126
1        mac_dl_cqi_offset    0.047816
2        mac_ul_snr_offset    0.036939
3            mac_dl_buffer    0.032483
4             mac_dl_brate    0.029135
5        phy_ul_pucch_rssi    0.020273
6               mac_ul_nok    0.018301
7                mac_dl_ok    0.017532
8               mac_dl_nok    0.012071
9                mac_ul_ok    0.011762
10              mac_ul_bsr    0.010576
11                 mac_phr    0.010470
12                mac_rnti    0.010125
13             mob_pattern    0.008521
14       phy_ul_pucch_sinr    0.006816
15  phy_ul_n_samples_pucch    0.005787
16        phy_dl_n_samples    0.004942
17              phy_ul_mcs    0.004924
18              mac_dl_cqi    0.004835
19       phy_ul_pusch_rssi    0.004740
20        phy_ul_n_samples    0.004530
21       phy_ul_pusch_sinr    0.004451
22             mac_nof_tti    0.004427
23              mac_dl_mcs    0.004255
24      phy_ul_turbo_iter